# Proyecto Hadoop: Resultados Presidenciales ONPE 2021 (Primera Vuelta)

Este notebook ejecuta un flujo **real** con Hadoop local (HDFS + MapReduce) usando el dataset oficial de ONPE: `presidencial-resultados-partidos.csv`.

## 1) Diagrama de diseño del sistema

```mermaid
flowchart LR
  onpeCsv[ONPECsvInput] --> normalizeStep[NormalizeColumnsStep]
  normalizeStep --> hdfsInput[HDFSInputPath]
  hdfsInput --> mapperNational[MapperCandidateVotes]
  hdfsInput --> mapperDept[MapperDeptCandidateVotes]
  mapperNational --> shuffleNational[ShuffleSortNational]
  mapperDept --> shuffleDept[ShuffleSortDepartment]
  shuffleNational --> reducerNational[ReducerSumVotes]
  shuffleDept --> reducerDept[ReducerSumVotes]
  reducerNational --> nationalResult[CandidateTotals]
  reducerDept --> deptResult[DepartmentCandidateTotals]
  nationalResult --> finalReport[WinnerAndPercentages]
```

## 2) Configuración mínima

In [ ]:
import csv
import subprocess
from pathlib import Path

PROJECT_DIR = Path('.').resolve()
RAW_FILE = PROJECT_DIR / 'data' / 'raw_votes' / 'presidencial-resultados-partidos.csv'
NORMALIZED_FILE = PROJECT_DIR / 'data' / 'raw_votes' / 'onpe_normalized_votes.csv'
OUTPUT_DIR = PROJECT_DIR / 'output'

NAMENODE = 'hadoop-elections-namenode'
RESOURCEMANAGER = 'hadoop-elections-resourcemanager'
HDFS_INPUT = '/elections/input'
HDFS_OUTPUT_CANDIDATE = '/elections/output/candidate_totals'
HDFS_OUTPUT_DEPARTMENT = '/elections/output/department_candidate_totals'
STREAMING_JAR = '/opt/hadoop-3.2.1/share/hadoop/tools/lib/hadoop-streaming-3.2.1.jar'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Proyecto: {PROJECT_DIR}')
print(f'Dataset ONPE: {RAW_FILE}')

In [ ]:
def run(cmd, check=True):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Fallo comando: {' '.join(cmd)}')
    return result

def docker_exec(container, command, check=True):
    return run(['docker', 'exec', container] + command, check=check)

def hdfs(args, check=True):
    return docker_exec(NAMENODE, ['hdfs', 'dfs'] + args, check=check)

def yarn_streaming(input_path, output_path, mapper, reducer):
    return docker_exec(
        RESOURCEMANAGER,
        [
            'yarn', 'jar', STREAMING_JAR,
            '-input', input_path,
            '-output', output_path,
            '-mapper', mapper,
            '-reducer', reducer,
            '-file', f'/mapreduce/{mapper}',
            '-file', f'/mapreduce/{reducer}',
        ],
    )

## 3) Prototipo local: normalizar dataset y subir a HDFS

Se crea un archivo simplificado con columnas:
- `departamento`
- `candidato`
- `total_votos`

Esto evita problemas por comas dentro del nombre del partido en el CSV original.

In [ ]:
with RAW_FILE.open(newline='', encoding='utf-8') as src, NORMALIZED_FILE.open('w', newline='', encoding='utf-8') as dst:
    reader = csv.DictReader(src)
    writer = csv.writer(dst)
    writer.writerow(['departamento', 'candidato', 'total_votos'])
    rows = 0
    for row in reader:
        writer.writerow([row['departamento'], row['candidato'], row['total_votos']])
        rows += 1

print(f'Filas normalizadas: {rows}')
print(f'Archivo: {NORMALIZED_FILE}')

In [ ]:
run(['docker', 'compose', 'up', '-d'])
hdfs(['-rm', '-r', '-f', HDFS_INPUT, HDFS_OUTPUT_CANDIDATE, HDFS_OUTPUT_DEPARTMENT], check=False)
hdfs(['-mkdir', '-p', HDFS_INPUT])
hdfs(['-put', f'/data/raw_votes/{NORMALIZED_FILE.name}', f'{HDFS_INPUT}/{NORMALIZED_FILE.name}'])
hdfs(['-ls', HDFS_INPUT])

## 4) Jobs MapReduce

In [ ]:
# Job 1: votos totales por candidato
yarn_streaming(HDFS_INPUT, HDFS_OUTPUT_CANDIDATE, 'mapper_candidate.sh', 'reducer_count.sh')
hdfs(['-cat', f'{HDFS_OUTPUT_CANDIDATE}/part-*'])

In [ ]:
# Job 2: votos por departamento y candidato
yarn_streaming(HDFS_INPUT, HDFS_OUTPUT_DEPARTMENT, 'mapper_city_candidate.sh', 'reducer_count.sh')
hdfs(['-cat', f'{HDFS_OUTPUT_DEPARTMENT}/part-*'])

## 5) Resultado final (reporte y archivos)

In [ ]:
def read_hdfs_lines(path):
    out = hdfs(['-cat', path]).stdout.strip()
    return [line for line in out.splitlines() if line]

candidate_lines = read_hdfs_lines(f'{HDFS_OUTPUT_CANDIDATE}/part-*')
candidate_totals = []
for line in candidate_lines:
    candidate, votes = line.split('\t')
    candidate_totals.append((candidate, int(votes)))

candidate_totals.sort(key=lambda x: x[1], reverse=True)
grand_total = sum(v for _, v in candidate_totals)
winner = candidate_totals[0][0]

candidate_file = OUTPUT_DIR / 'candidate_totals.csv'
with candidate_file.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['candidate', 'total_votes', 'percentage'])
    for cand, votes in candidate_totals:
        w.writerow([cand, votes, round((votes / grand_total) * 100, 2)])

dept_lines = read_hdfs_lines(f'{HDFS_OUTPUT_DEPARTMENT}/part-*')
dept_file = OUTPUT_DIR / 'department_candidate_totals.csv'
with dept_file.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['departamento', 'candidate', 'total_votes'])
    for line in dept_lines:
        key, votes = line.split('\t')
        dept, cand = key.split('|', 1)
        w.writerow([dept, cand, int(votes)])

summary_file = OUTPUT_DIR / 'winner_summary.txt'
summary_text = (
    f'Ganador nacional: {winner}\n'
    f'Total votos válidos agregados: {grand_total}\n'
    'Top 5 candidatos:\n' + '\n'.join(
        f'  {c}: {v} ({round((v / grand_total) * 100, 2)}%)'
        for c, v in candidate_totals[:5]
    )
)
summary_file.write_text(summary_text, encoding='utf-8')

print('=== RESULTADO FINAL ===')
print(summary_text)
print(f'Archivos: {candidate_file.name}, {dept_file.name}, {summary_file.name}')

## 6) Preguntas de discusión

- **¿Por qué HDFS divide archivos en bloques?** Para distribuir almacenamiento/procesamiento y tolerar fallos con réplicas.
- **¿Responsabilidad del mapper?** Transformar cada fila de entrada en pares clave-valor (`candidato -> total_votos` o `departamento|candidato -> total_votos`).
- **¿Responsabilidad del reducer?** Agregar por clave y producir la suma final de votos.
- **¿Cuándo Hadoop vs script Python?** Hadoop es mejor con volumen y paralelismo distribuidos; Python simple basta para datasets pequeños en una sola máquina.